# E006 — Offline evaluation of ranking policies

**Hypotheses:** H11, H6 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Would ranking by the model's pCTR serve more clicked ads than the policy that produced the logs, and what do safety gates and exploration cost?

**Method:** Reconstruct candidate sets per publisher x hour on the test days, take each creative's share of the cell as its logging propensity, re-score every (impression, candidate) pair with the shipped bundle, and estimate each policy with IPS, SNIPS and doubly robust estimators plus paired hour-block bootstrap lifts.

In [1]:
from IPython.display import Markdown

from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
from charade.analysis import policy_eval
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
policy_eval.run(ctx.settings, ctx.data_dir, report=ctx.reports / "policy" / "ope.md")
Markdown((ctx.reports / "policy" / "ope.md").read_text())

# Off-policy evaluation on the test split (generated by `uv run poe ope`)

DR uses an independent LightGBM reward model (trained before the last training day), not the evaluated policy's own pCTR.

Evaluated impressions: 102,874 of 127,406 test rows (80.7%); cells: 5,484; mean candidates per cell: 4.13

| Policy | Estimator | CTR | 95 % CI | ESS | Max weight |
|---|---|---|---|---|---|
| logging (observed) | on-policy | 0.1723 | [0.1649, 0.1795] | 102,874 | 1.0 |
| uniform random | ips | 0.1905 | [0.1814, 0.1987] | 17,149 | 196.9 |
| uniform random | snips | 0.1760 | [0.1694, 0.1819] | 17,149 | 196.9 |
| uniform random | dr | 0.1716 | [0.1643, 0.1782] | 17,149 | 196.9 |
| greedy pCTR (no gates) | ips | 0.2023 | [0.1864, 0.2164] | 4,059 | 445.0 |
| greedy pCTR (no gates) | snips | 0.1968 | [0.1832, 0.2099] | 4,059 | 445.0 |
| greedy pCTR (no gates) | dr | 0.1864 | [0.1736, 0.1989] | 4,059 | 445.0 |
| shipped policy (gates + 5% exploration) | ips | 0.1944 | [0.1797, 0.2063] | 4,395 | 425.7 |
| shipped policy (gates + 5% exploration) | snips | 0.1941 | [0.1809, 0.2063] | 4,395 | 425.7 |
| shipped policy (gates + 5% exploration) | dr | 0.1846 | [0.1722, 0.1963] | 4,395 | 425.7 |

## Lift over the logging policy (paired hour-block bootstrap)

| Policy | SNIPS lift [95 % CI] | DR lift [95 % CI] |
|---|---|---|
| uniform random | +0.0037 [-0.0000, +0.0071] | -0.0007 [-0.0043, +0.0024] |
| greedy pCTR (no gates) | +0.0245 [+0.0140, +0.0359] | +0.0141 [+0.0038, +0.0253] |
| shipped policy (gates + 5% exploration) | +0.0218 [+0.0112, +0.0325] | +0.0123 [+0.0020, +0.0220] |


## Reading
- Uniform random lands near the logging CTR, as it should when logging is frequency-proportional.
- Both the ungated greedy and the shipped policy have DR lifts whose intervals exclude zero; the
  shipped policy pays ~0.16 pp for brand safety, caps and exploration.
- The intervals cover sampling noise under the reconstruction assumptions, not bias from them.
- H6 shows up as the frequency-cap gate and as `log_user_campaign_imps` in the model.

## Conclusion
Under reconstructed candidate sets and frequency-share propensities, with DR using an independent LightGBM reward model, the shipped policy's estimated lift over logging is +1.23 pp CTR [+0.20, +2.20] (ungated greedy +1.41 pp; validation day +2.08 pp); ESS ~4 % of rows; a demonstration of the evaluation, not a measured production lift.

**Decision:** Ship greedy + gates + 5 % exploration from a closed-form distribution, and log every candidate's exact propensity so the next evaluation does not need reconstruction.